### Assumptions of one way [ANOVA](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html)
1. The samples are independent.
2. Each sample is from a normally distributed population.
3. The population standard deviations of the groups are all equal. This property is known as homoscedasticity



In [1]:
from scipy.stats import friedmanchisquare
import pandas as pd

In [2]:
df = pd.read_csv('../results.csv')


In [3]:
chr20 = df[df['Chromosome'] == 'chr20']

coverages = df['Coverage'].unique().tolist()
tools = df['Tool'].unique().tolist()

datasets = {}

for cov in coverages:
    for tool in tools:
        subset = chr20[
            (chr20['Tool'] == tool) &
            (chr20['Coverage'] == cov)
        ]

        datasets[(cov, tool)] = subset


In [4]:
def summary(column):
    
    summary_rows = []
    for cov in coverages:
        for tool in ["bcftools", "gatk", "freebayes"]:
            vals = pd.to_numeric(datasets[(cov, tool)].sort_values("Seed")[column])
            summary_rows.append(
                {
                    "Coverage": cov,
                    "Tool": tool,
                    f"Median {column}": vals.median(),
                    "Min": vals.min(),
                    "Max": vals.max(),
                }
            )

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_markdown(index=False))

summary('Sensitivity')

KeyError: (10, 'bcftools')

In [16]:
tools = ['GATK HaplotypeCaller',  'FreeBayes', 'BCFtools']

gatk = df[df.loc[:, 'Tool'] == tools[0]]
fb = df[df.loc[:, 'Tool'] == tools[1]]
bcf = df[df.loc[:, 'Tool'] == tools[2]]

gatk_val = gatk.loc[:, 'F-measure'].to_list()
fb_val = fb.loc[:, 'F-measure'].to_list()
bcf_val = bcf.loc[:, 'F-measure'].to_list()

res = friedmanchisquare(gatk_val, fb_val, bcf_val)
print(res.statistic)
print(res.pvalue)

94.75
2.66255703625835e-21


In [18]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

tools = ['GATK HaplotypeCaller', 'FreeBayes', 'BCFtools']

results = []

for t1, t2 in combinations(tools, 2):
    x = df[df['Tool'] == t1]['Sensitivity'].to_numpy()
    y = df[df['Tool'] == t2]['Sensitivity'].to_numpy()

    stat, p = wilcoxon(x, y)

    results.append((t1, t2, stat, p))

# Holm correction
p_values = [r[3] for r in results]
reject, corrected_p, _, _ = multipletests(
    p_values, method='holm'
)

for (t1, t2, stat, p), p_adj, sig in zip(
    results, corrected_p, reject
):
    print(f'{t1} vs {t2}')
    print(f'  W = {stat:.3f}')
    print(f'  Raw p = {p:.4e}')
    print(f'  Adjusted p = {p_adj:.4e}')
    print(f'  Significant: {sig}')

GATK HaplotypeCaller vs FreeBayes
  W = 0.000
  Raw p = 9.4608e-23
  Adjusted p = 2.8382e-22
  Significant: True
GATK HaplotypeCaller vs BCFtools
  W = 0.000
  Raw p = 1.3460e-22
  Adjusted p = 2.8382e-22
  Significant: True
FreeBayes vs BCFtools
  W = 0.000
  Raw p = 9.4656e-23
  Adjusted p = 2.8382e-22
  Significant: True


In [ ]:
import numpy as np
for t1, t2 in combinations(tools, 2):
    x = df[df['Tool'] == t1]['Sensitivity'].to_numpy()
    y = df[df['Tool'] == t2]['Sensitivity'].to_numpy()

    diff = x - y

    print(f"\n{t1} vs {t2}")
    print(f"Median difference: {np.median(diff):.6f}")
    print(f"Mean difference: {np.mean(diff):.6f}")
    print(f"First caller wins: {np.sum(diff > 0)}")
    print(f"Second caller wins: {np.sum(diff < 0)}")
    print(f"Ties: {np.sum(diff == 0)}")